In [ ]:
# Imports & Setup

import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("All imports successful!")

In [ ]:
# Load Dataset

columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('/content/training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=columns)

# Quick look
print("Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Explore & Clean Labels
# Keep only the columns we need
df = df[['target', 'text']]

# The dataset uses 0 = negative, 4 = positive
# Let's convert 4 → 1 so labels are 0 and 1
df['target'] = df['target'].replace(4, 1)

print("Label distribution:")
print(df['target'].value_counts())
plt.figure(figsize=(6, 4))
sns.countplot(x='target', data=df, palette=['#e74c3c', '#2ecc71'])
plt.xticks([0, 1], ['Negative (0)', 'Positive (1)'])
plt.title('Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Sample the Data
# 1.6M rows is too large to train quickly
# 200,000 rows (100k negative + 100k positive)
# This keeps the dataset balanced and trains faster

df_sample = df.groupby('target').apply(
    lambda x: x.sample(100000, random_state=42)
).reset_index(drop=True)

print("Sample shape:", df_sample.shape)
print("\nLabel distribution in sample:")
print(df_sample['target'].value_counts())

In [ ]:
# Text Preprocessing
# In this step we clean raw tweets so the model only sees meaningful words

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_tweet(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # 3. Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # 4. Remove hashtags
    text = re.sub(r'#\w+', '', text)
    # 5. Remove numbers
    text = re.sub(r'\d+', '', text)
    # 6. Remove punctuation and special characters
    text = re.sub(r'[^a-z\s]', '', text)
    # 7. Remove extra whitespace
    text = text.strip()
    # 8. Tokenize, remove stopwords, lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return ' '.join(words)

print("Cleaning tweets... this may take 2-3 minutes ⏳")
df_sample['clean_text'] = df_sample['text'].apply(clean_tweet)
print("Done!")

# Show before and after
print("\nBefore:", df_sample['text'].iloc[0])
print("After: ", df_sample['clean_text'].iloc[0])

In [ ]:
# TF-IDF Feature Extraction
# TF-IDF converts text into numbers the model can understand
# TF = how often a word appears in a tweet
# IDF = how rare/important that word is across all tweets
# Words that appear everywhere (like "the") get low scores
# Unique, meaningful words get high scores

X = df_sample['clean_text']
y = df_sample['target']

# Split into train and test BEFORE vectorizing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# TF-IDF Vectorizer
# max_features = only keep top 50,000 most important words
# ngram_range = (1,2) means consider single words AND word pairs
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f"\n TF-IDF matrix shape: {X_train_tfidf.shape}")
print("(rows = tweets, columns = unique word features)")

In [ ]:
# Train & Compare Models
# We train 3 models and compare their performance
# This tells us which algorithm works best for our data

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'Linear SVM': LinearSVC(max_iter=1000)
}

results = {}

for name, model in models.items():
    print(f"Training {name}...")

    # Train
    model.fit(X_train_tfidf, y_train)

    # Predict
    y_pred = model.predict(X_test_tfidf)

    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f" {name} Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    print(classification_report(y_test, y_pred,
                                target_names=['Negative', 'Positive']))
    print("-" * 60)

print("\n Best Model:", max(results, key=results.get))

In [ ]:
# Visualize Model Comparison

# Bar chart comparing all 3 models
plt.figure(figsize=(8, 5))
bars = plt.bar(results.keys(),
               [v*100 for v in results.values()],
               color=['#3498db', '#e74c3c', '#2ecc71'],
               width=0.5)

# Add accuracy labels on top of bars
for bar, acc in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.3,
             f'{acc*100:.2f}%',
             ha='center', fontsize=12, fontweight='bold')

plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy (%)')
plt.ylim(70, 82)
plt.tight_layout()
plt.show()

# Confusion matrix for best model (Logistic Regression)
lr_model = models['Logistic Regression']
y_pred_lr = lr_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, y_pred_lr)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix — Logistic Regression', fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Word Clouds
# Visual representation of most frequent words
# Bigger word = appears more often in that sentiment

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, (label, sentiment) in enumerate([(0, 'Negative'), (1, 'Positive')]):
    text = ' '.join(df_sample[df_sample['target'] == label]['clean_text'])

    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap='Reds' if label == 0 else 'Greens',
        max_words=100
    ).generate(text)

    axes[i].imshow(wordcloud, interpolation='bilinear')
    axes[i].axis('off')
    axes[i].set_title(f'{sentiment} Tweets — Most Common Words',
                       fontsize=14, fontweight='bold',
                       color='#e74c3c' if label == 0 else '#27ae60')

plt.tight_layout()
plt.show()

In [ ]:
# Predict on New Tweets
# Now we use our trained model to predict sentiment
# on brand new tweets it has never seen before

def predict_sentiment(tweet):
    cleaned = clean_tweet(tweet)
    vectorized = tfidf.transform([cleaned])
    prediction = lr_model.predict(vectorized)[0]
    sentiment = 'Positive' if prediction == 1 else 'Negative'
    return sentiment

# Test with your own examples!
test_tweets = [
    "I love this sunny day, feeling amazing!",
    "This is the worst day of my life, everything went wrong",
    "Just had the best coffee ever, so happy!",
    "I hate Mondays, so tired and bored",
    "My team won the match, absolutely incredible!",
    "Missing my friends so much, feeling lonely"
]

print("        SENTIMENT PREDICTIONS ON NEW TWEETS")
print("=" * 50)

for tweet in test_tweets:
    sentiment = predict_sentiment(tweet)
    print(f"\nTweet: {tweet}")
    print(f"Sentiment: {sentiment}")
    print("-" * 55)

In [ ]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
!python -m textblob.download_corpora

In [ ]:
# Emotion Detection (Manual Lexicon Approach)
# We define emotion keywords ourselves
# This is transparent, understandable, and works perfectly

emotion_lexicon = {
    'joy':          ['happy', 'love', 'amazing', 'wonderful', 'great',
                     'fantastic', 'excited', 'glad', 'yay', 'awesome',
                     'best', 'incredible', 'brilliant', 'thank'],
    'sadness':      ['sad', 'miss', 'lonely', 'cry', 'unhappy', 'miserable',
                     'depressed', 'heartbroken', 'disappointed', 'sorry'],
    'anger':        ['hate', 'angry', 'furious', 'annoyed', 'mad', 'worst',
                     'stupid', 'idiot', 'terrible', 'horrible', 'damn'],
    'fear':         ['scared', 'afraid', 'worried', 'nervous', 'anxious',
                     'terrified', 'panic', 'dread', 'frightened'],
    'surprise':     ['wow', 'omg', 'unbelievable', 'shocking', 'unexpected',
                     'incredible', 'amazing', 'whoa', 'really'],
    'anticipation': ['hope', 'excited', 'looking', 'forward', 'soon',
                     'tomorrow', 'waiting', 'expect', 'cant wait']
}

def detect_emotions(tweet):
    tweet_lower = tweet.lower()
    words = tweet_lower.split()
    scores = {emotion: 0 for emotion in emotion_lexicon}

    for emotion, keywords in emotion_lexicon.items():
        for word in words:
            if word in keywords:
                scores[emotion] += 1

    # Filter out zero scores
    scores = {k: v for k, v in scores.items() if v > 0}

    if not scores:
        return {}, 'neutral'

    top_emotion = max(scores, key=scores.get)
    return scores, top_emotion

# Emotion emojis
emotion_emoji = {
    'joy': '😊', 'sadness': '😢', 'anger': '😡',
    'fear': '😨', 'surprise': '😲', 'anticipation': '🤩',
    'neutral': '😶'
}

print("         EMOTION DETECTION ON NEW TWEETS")
print("=" * 50)

for tweet in test_tweets:
    scores, top = detect_emotions(tweet)
    emoji = emotion_emoji.get(top, '😐')
    print(f"\nTweet: {tweet}")
    print(f"Emotions detected: {scores}")
    print(f"Dominant Emotion: {emoji} {top.upper()}")
    print("-" * 60)

In [ ]:
# Combined Sentiment + Emotion Prediction
# This combines both our models into one powerful function
# Sentiment = positive/negative (ML model)
# Emotion = specific feeling (lexicon)

def analyze_tweet(tweet):
    # Step 1: Sentiment
    cleaned = clean_tweet(tweet)
    vectorized = tfidf.transform([cleaned])
    sentiment_pred = lr_model.predict(vectorized)[0]
    sentiment = 'Positive' if sentiment_pred == 1 else 'Negative'

    # Step 2: Emotion
    scores, top_emotion = detect_emotions(tweet)
    emoji = emotion_emoji.get(top_emotion, '😐')

    # Step 3: Display
    print(f"📝 Tweet     : {tweet}")
    print(f"💬 Sentiment : {sentiment}")
    print(f"❤️  Emotion   : {emoji} {top_emotion.upper()}")
    if scores:
        print(f"📊 Breakdown : {scores}")
    print("-" * 50)

# Test it!
print("     COMBINED SENTIMENT + EMOTION ANALYSIS")
print("=" * 50)

custom_tweets = [
    "I love this sunny day, feeling amazing!",
    "This is the worst day of my life, everything went wrong",
    "Just had the best coffee ever, so happy!",
    "I hate Mondays, so tired and bored",
    "My team won the match, absolutely incredible!",
    "Missing my friends so much, feeling lonely"
]

for tweet in custom_tweets:
    analyze_tweet(tweet)

In [ ]:
# Sentiment Confidence Score
# Instead of just Positive/Negative, we show HOW confident
# the model is about its prediction (as a probability %)
# This gives much more nuanced insight than a binary label

def analyze_with_confidence(tweet):
    cleaned = clean_tweet(tweet)
    vectorized = tfidf.transform([cleaned])

    # predict_proba gives probability for each class
    probabilities = lr_model.predict_proba(vectorized)[0]
    neg_conf = probabilities[0] * 100
    pos_conf = probabilities[1] * 100

    sentiment = 'Positive' if pos_conf > neg_conf else 'Negative'
    confidence = max(pos_conf, neg_conf)

    # Confidence level label
    if confidence >= 80:
        level = "Very Confident"
    elif confidence >= 65:
        level = "Confident"
    else:
        level = "Uncertain"

    print(f"📝 Tweet      : {tweet}")
    print(f"💬 Sentiment  : {sentiment}")
    print(f"📊 Confidence : {confidence:.1f}% ({level})")
    print(f"   Negative: {neg_conf:.1f}% | Positive: {pos_conf:.1f}%")
    print("-" * 60)

print("      SENTIMENT ANALYSIS WITH CONFIDENCE SCORES")
print("=" * 60)

test_tweets_confidence = [
    "I absolutely love everything about today!",
    "This is okay I guess",
    "I hate this so much, worst experience ever",
    "Not sure how I feel about this",
    "Best day of my life, so grateful!",
    "Things could be better honestly"
]

for tweet in test_tweets_confidence:
    analyze_with_confidence(tweet)

In [ ]:
# Emotion Distribution Across Dataset
# Run emotion detection on 2000 tweets from our dataset
# and visualize which emotions dominate social media overall

from collections import Counter

print("Analyzing emotions across 2000 tweets... ⏳")

# Sample 1000 positive and 1000 negative tweets
sample_pos = df_sample[df_sample['target'] == 1]['text'].sample(1000, random_state=42)
sample_neg = df_sample[df_sample['target'] == 0]['text'].sample(1000, random_state=42)

def get_top_emotion(tweet):
    _, top = detect_emotions(tweet)
    return top

# Get emotions for both
pos_emotions = [get_top_emotion(t) for t in sample_pos]
neg_emotions = [get_top_emotion(t) for t in sample_neg]

pos_counts = Counter(pos_emotions)
neg_counts = Counter(neg_emotions)

print(" Done!")
print("\nPositive tweet emotions:", dict(pos_counts))
print("Negative tweet emotions:", dict(neg_counts))

# Plot side by side
emotions = ['joy', 'sadness', 'anger', 'fear', 'surprise', 'anticipation', 'neutral']

pos_vals = [pos_counts.get(e, 0) for e in emotions]
neg_vals = [neg_counts.get(e, 0) for e in emotions]

x = range(len(emotions))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar([i - width/2 for i in x], pos_vals, width,
                label='Positive Tweets', color='#2ecc71')
bars2 = ax.bar([i + width/2 for i in x], neg_vals, width,
                label='Negative Tweets', color='#e74c3c')

ax.set_xlabel('Emotion')
ax.set_ylabel('Number of Tweets')
ax.set_title('Emotion Distribution: Positive vs Negative Tweets',
             fontsize=14, fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels([e.capitalize() for e in emotions])
ax.legend()

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(int(bar.get_height())), ha='center', fontsize=9)

plt.tight_layout()
plt.show()